# E009: better pair features + stage-2 stacking + per-S1 expected-F selection + count matching

E008 (E007 blocker): CV macro F0.5 **0.9507**. Precision 0.980, recall 0.912. Oracle on the same
candidates: 0.994, so ~7 pt of true pairs are retrieved but scored too low. Stress (country held out):
India 0.844, US 0.926.

| step | change | why |
|---|---|---|
| CV (v3) | + fuzzy house-number features, + `conj_cos` (token-pair cosine) as a feature; fold models saved | truncated/padded house numbers; the model only saw conj's rank |
| selection | global threshold **vs** per-S1 expected-F0.5 on calibrated p (explicit "predict nothing" option) | macro F0.5 is per entity; singleton F0.5 was 0.914 |
| stack | stage 2 on out-of-fold p: probability context in the S1 + sibling agreement (is this record like the S1's confident other matches?) | native-script / rewritten records resemble their siblings |
| stress | + count matching: shift an unseen country's logits so matches-per-S1 equals the out-of-fold value (train: 3.46 per S1, 5.6% singletons in BOTH countries) | stress India was 0.844 |
| predict | test = average of the 3 fold models (+ stage 2), then 3 submissions: A plain, B count-match France only, C count-match all countries | |

**Run in the SAME Kaggle notebook as E004/E008** (caches: prepared data, E007 train/test candidates, the
E008 feature matrix). GPU not needed (test candidates are cached). Internet On. Persistence Files.
Everything caches; *Run all* resumes.

In [ ]:
# 1. Config
EXP       = "20260925-E009-v3-stack"
E008_EXP  = "20260925-E008-E007"
E007_SPEC = "base,conj:20,bm25:5,bge_native:5,name_noaddr:5"
CV_FRAC   = 0.1      # same S1 sample as E008 -> its cached v2 feature matrix is reused
LR        = 0.1
REPO      = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR  = "/kaggle/working/ber"
WORK      = "/kaggle/working/work"

In [ ]:
# 2. Dataset, validator, caches
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found"
DATA = os.path.dirname(os.path.dirname(hits[0]))
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
def reuse(pattern, dest_dir):
    for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        d = os.path.join(dest_dir, os.path.basename(p))
        if not os.path.exists(d):
            os.makedirs(dest_dir, exist_ok=True); os.symlink(p, d); print("reusing", p)
for split in ("train", "test"):
    reuse(f"work/prepared/{split}/*.parquet", f"{WORK}/prepared/{split}")
    reuse(f"work/{split}/*.parquet", f"{WORK}/{split}")
print("DATA =", DATA, "| VALIDATOR =", VALIDATOR)
!ls -la {WORK}/train {WORK}/test; free -g; nproc

In [ ]:
# 3. Code, dependencies, tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper
import subprocess, time, json
import pandas as pd
def ber(cmd, *extra):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", EXP, "--cv-frac", str(CV_FRAC),
            "--lr", str(LR), "--df-cap", "2500", "--channels", E007_SPEC, "--feat", "v3", *map(str, extra)]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min")
X = lambda name: f"{WORK}/experiments/{EXP}/{name}"

In [ ]:
# 4. Stage-1 CV with v3 features (3 folds, fold models saved, selection study, stress + count matching)
if not os.path.exists(X("cv_metrics.json")):
    ber("cv", "--folds", 3)
cv = json.load(open(X("cv_metrics.json")))
keys = [k for k in cv if k.startswith(("cv", "stress")) and isinstance(cv[k], dict)]
display(pd.DataFrame({k: cv[k] for k in keys}).T[["macro_f05", "micro_precision", "micro_recall", "f05_singletons",
                                                  "f05_nonsingletons"]].round(4))
print("threshold", cv["threshold"], "| best selection", cv["selection"]["best"], "->", round(cv["selection"]["macro_f05"], 5),
      "| selected/S1", round(cv["selected_per_s1_oof"], 3))
print("top features", list(cv["feature_gain_top"].items())[:12])

In [ ]:
# 5. Stage 2 (stacking) + selection study on stage-1 and stage-2 out-of-fold p
if not os.path.exists(X("stack_metrics.json")):
    ber("stack", "--folds", 3)
sm = json.load(open(X("stack_metrics.json")))
e8 = json.load(open(f"{WORK}/experiments/{E008_EXP}/cv_metrics.json"))
rows = [("E008 (v2, threshold)", e8["cv"]["macro_f05"]),
        ("E009 stage 1 (v3, threshold)", cv["cv"]["macro_f05"]),
        ("E009 stage 1 best selection " + json.dumps(sm["stage1"]["best"]), sm["stage1"]["macro_f05"]),
        ("E009 stage 2 best selection " + json.dumps(sm["stage2"]["best"]), sm["stage2"]["macro_f05"])]
display(pd.DataFrame(rows, columns=["variant", "CV macro F0.5"]).round(5))
print("use_stack:", sm["use_stack"])
display(pd.DataFrame({k: sm[k] for k in sm if k.startswith("chosen")}).T.round(4))
print("stage-2 top features", list(sm["stack_feature_gain_top"].items())[:12])

In [ ]:
# 6. Test prediction (fold-model average, + stage 2 if chosen) and three submissions
#    A = plain, B = count-match unseen countries only (France), C = count-match every country
OUTS = {"A": ("none", "/kaggle/working/output_E009_A"), "B": ("unseen", "/kaggle/working/output_E009_B"),
        "C": ("all", "/kaggle/working/output_E009_C")}
if not os.path.exists(X("test_pairs_proba.parquet")):
    ber("predict", "--out", OUTS["A"][1])
for k, (cm, od) in OUTS.items():
    if not os.path.exists(f"{od}/matching_results.tsv"):
        ber("select", "--count-match", cm, "--out", od)
    if VALIDATOR:
        !python {VALIDATOR} --matching {od}/matching_results.tsv --candidate {od}/candidate_pairs.tsv --test-dir {DATA}/test --check-ids | tail -2
    info = json.load(open(f"{od}/selection_info.json"))
    print(k, "count_match =", cm, "| delta", info["delta"])
    display(pd.DataFrame(info["per_country"]).T.round(3))

In [ ]:
# 7. Bundle: send /kaggle/working/E009_results.tgz back (metrics + a 60k-row error sample; no submissions inside)
import shutil, tarfile
B = "/kaggle/working/E009_results"
os.makedirs(B, exist_ok=True)
for f in ("cv_metrics.json", "stack_metrics.json", "oof_errors.parquet"):
    if os.path.exists(X(f)):
        shutil.copy(X(f), f"{B}/{f}")
for k, (_, od) in OUTS.items():
    if os.path.exists(f"{od}/selection_info.json"):
        shutil.copy(f"{od}/selection_info.json", f"{B}/selection_info_{k}.json")
with tarfile.open("/kaggle/working/E009_results.tgz", "w:gz") as t:
    t.add(B, arcname="E009_results")
print(sorted(os.listdir(B)), os.path.getsize("/kaggle/working/E009_results.tgz") // 1024, "KiB")